In [1]:
from __future__ import annotations


class Endereco:
    def __init__(self, rua: str, numero: str, bairro: str, cidade: str, estado: str, cep: str):
        self.rua = rua
        self.numero = numero
        self.bairro = bairro
        self.cidade = cidade
        self.estado = estado
        self.cep = cep

    def atualizar(self, **dados) -> None:
        for campo, valor in dados.items():
            if not hasattr(self, campo):
                raise AttributeError(f"Campo inválido: {campo}")
            setattr(self, campo, valor)

    def formatar(self) -> str:
        return (
            f"{self.rua}, {self.numero} - {self.bairro}, "
            f"{self.cidade}/{self.estado} - CEP {self.cep}"
        )

    def __str__(self) -> str:
        return self.formatar()


class Aluno:
    """Agregação com Endereco: o endereço nasce com o aluno, mas sobrevive a ele."""

    def __init__(self, nome: str, matricula: str, data_nascimento: str, **dados_endereco):
        self.nome = nome
        self.matricula = matricula
        self.data_nascimento = data_nascimento
        self.endereco: Endereco | None = Endereco(**dados_endereco)
        self.ativo = True

    def atualizar_endereco(self, **dados) -> None:
        if self.endereco is None:
            raise ValueError("Aluno removido não possui endereço vinculado.")
        self.endereco.atualizar(**dados)

    def remover(self) -> Endereco | None:
        endereco = self.endereco
        self.endereco = None
        self.ativo = False
        return endereco

    def __str__(self) -> str:
        return f"Aluno {self.nome} (matrícula {self.matricula})"


class SalaDeAula:
    """Parte de Escola: só deve ser criada por Escola.adicionar_sala()."""

    def __init__(self, numero: str, capacidade: int, escola: Escola):
        self.numero = numero
        self.capacidade = capacidade
        self._escola = escola
        self.disponivel = True

    @property
    def escola(self) -> Escola:
        return self._escola

    def reservar(self) -> bool:
        if not self.disponivel:
            return False
        self.disponivel = False
        return True

    def liberar(self) -> None:
        self.disponivel = True

    def __str__(self) -> str:
        return f"Sala {self.numero} (capacidade {self.capacidade}) - {self._escola.nome}"


class Professor:
    """Associação N:N com Escola: ambos existem de forma independente."""

    def __init__(self, nome: str, registro: str, disciplina: str):
        self.nome = nome
        self.registro = registro
        self.disciplina = disciplina
        self.escolas: list[Escola] = []

    def vincular_escola(self, escola: Escola) -> None:
        if escola not in self.escolas:
            self.escolas.append(escola)
        if self not in escola.professores:
            escola.professores.append(self)

    def desvincular_escola(self, escola: Escola) -> None:
        if escola in self.escolas:
            self.escolas.remove(escola)
        if self in escola.professores:
            escola.professores.remove(self)

    def lecionar(self, escola: Escola, turma: str) -> str:
        if escola not in self.escolas:
            raise ValueError(f"{self.nome} não leciona em {escola.nome}.")
        return f"{self.nome} leciona {self.disciplina} para a turma {turma} em {escola.nome}."

    def __str__(self) -> str:
        return f"Professor {self.nome} ({self.disciplina})"


class Escola:
    """Composição com SalaDeAula: fechar a escola destrói suas salas."""

    def __init__(self, nome: str, cnpj: str):
        self.nome = nome
        self.cnpj = cnpj
        self.salas: list[SalaDeAula] = []
        self.professores: list[Professor] = []
        self.aberta = True

    def adicionar_sala(self, numero: str, capacidade: int) -> SalaDeAula:
        sala = SalaDeAula(numero, capacidade, self)
        self.salas.append(sala)
        return sala

    def remover_sala(self, numero: str) -> None:
        self.salas = [s for s in self.salas if s.numero != numero]

    def contratar_professor(self, professor: Professor) -> None:
        professor.vincular_escola(self)

    def fechar(self) -> None:
        self.salas.clear()
        for professor in list(self.professores):
            professor.desvincular_escola(self)
        self.aberta = False

    def capacidade_total(self) -> int:
        return sum(s.capacidade for s in self.salas)

    def __str__(self) -> str:
        return f"Escola {self.nome} ({len(self.salas)} salas, {len(self.professores)} professores)"


def main() -> None:
    escola_a = Escola("Escola Municipal Alfa", "00.000.000/0001-00")
    escola_b = Escola("Colégio Beta", "11.111.111/0001-11")

    escola_a.adicionar_sala("101", 30)
    escola_a.adicionar_sala("102", 25)
    escola_b.adicionar_sala("A1", 40)

    prof = Professor("Maria Souza", "P-123", "Matemática")
    escola_a.contratar_professor(prof)
    escola_b.contratar_professor(prof)

    aluno = Aluno(
        "João Lima", "2026001", "2012-03-15",
        rua="Rua das Flores", numero="45", bairro="Centro",
        cidade="Parnaíba", estado="PI", cep="64200-000",
    )

    print(escola_a)
    print(escola_b)
    print(prof.lecionar(escola_a, "7º A"))
    print(aluno, "-", aluno.endereco)

    print("\n--- Fechando a Escola Alfa (composição) ---")
    escola_a.fechar()
    print(f"Salas da Alfa: {len(escola_a.salas)}")
    print(f"Professor continua existindo: {prof} | escolas: {[e.nome for e in prof.escolas]}")

    print("\n--- Removendo o aluno (agregação) ---")
    endereco_sobrevivente = aluno.remover()
    print(f"Aluno ativo: {aluno.ativo} | endereço do aluno: {aluno.endereco}")
    print(f"Endereço preservado para relatórios: {endereco_sobrevivente}")


if __name__ == "__main__":
    main()

Escola Escola Municipal Alfa (2 salas, 1 professores)
Escola Colégio Beta (1 salas, 1 professores)
Maria Souza leciona Matemática para a turma 7º A em Escola Municipal Alfa.
Aluno João Lima (matrícula 2026001) - Rua das Flores, 45 - Centro, Parnaíba/PI - CEP 64200-000

--- Fechando a Escola Alfa (composição) ---
Salas da Alfa: 0
Professor continua existindo: Professor Maria Souza (Matemática) | escolas: ['Colégio Beta']

--- Removendo o aluno (agregação) ---
Aluno ativo: False | endereço do aluno: None
Endereço preservado para relatórios: Rua das Flores, 45 - Centro, Parnaíba/PI - CEP 64200-000
